# MicroPlant — Model Training

This notebook trains the MicroPlant model for plant disease classification.

We explore:
- Baseline training (MicroPlant)
- Teacher model (ResNet18)
- Knowledge Distillation (KD)

The goal is to build an accurate yet lightweight model suitable for edge deployment.

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.preprocessing import get_dataloaders
from src.architectures import get_microplant, get_teacher_model
from src.training import train_model, validate
from src.utils import count_parameters, count_model_bytes

import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="torch.utils.data.dataloader")

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

We define training parameters and device setup.

In [2]:
DATA_DIR = "../data/MicroPlantDataset"

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

BATCH_SIZE = 64
EPOCHS = 10
LR = 0.001

print(f"using : {DEVICE}")

using : cpu


## Load Dataset

We use the predefined data pipeline with augmentation and proper splitting.

In [3]:
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    DATA_DIR, batch_size=BATCH_SIZE
)

print("Classes:", class_names)

Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy']


## Baseline Model (MicroPlant)
(see model in src/architecture.py)

We first train the MicroPlant model without any distillation.

This serves as a baseline for comparison.

In [4]:
baseline_model = get_microplant(num_classes=len(class_names)).to(DEVICE)

print("Parameters:", count_parameters(baseline_model))
count_model_bytes(baseline_model)

Parameters: 8804
Model Weights: 35,216 bytes
Model Buffers: 2,648 bytes
Total Size: 36.98 KB


37864

In [5]:
baseline_model = train_model(
    baseline_model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    save_name="../models/microplant_baseline",
    device=DEVICE
)

Training: 100%|██████████| 41/41 [00:35<00:00,  1.17it/s]


Epoch 1 | Train Loss 1.2183 F1 0.46 | Val Loss 1.2480 F1 0.4101


Training: 100%|██████████| 41/41 [00:32<00:00,  1.28it/s]


Epoch 2 | Train Loss 0.8509 F1 0.64 | Val Loss 0.6523 F1 0.6775


Training: 100%|██████████| 41/41 [00:31<00:00,  1.31it/s]


...

Epoch 8 | Train Loss 0.2236 F1 0.93 | Val Loss 0.1417 F1 0.9540


Training: 100%|██████████| 41/41 [00:32<00:00,  1.27it/s]


Epoch 9 | Train Loss 0.1941 F1 0.93 | Val Loss 0.1351 F1 0.9538


Training: 100%|██████████| 41/41 [00:29<00:00,  1.39it/s]


Epoch 10 | Train Loss 0.2184 F1 0.94 | Val Loss 0.1282 F1 0.9434


In [15]:
baseline_loss, baseline_f1 = validate(baseline_model, test_loader, device=DEVICE)

print(f"Baseline F1: {baseline_f1:.4f}")

Baseline F1: 0.9344


## Teacher Model (ResNet18)
(see model in src/architecture.py)

We fine-tune a larger model to act as a teacher. (only the last layer is activated for optimzer step)

This model provides strong supervision for distillation.

In [6]:
teacher_model = get_teacher_model(num_classes=len(class_names)).to(DEVICE)

print("Teacher parameters:", count_parameters(teacher_model))
count_model_bytes(teacher_model)

Teacher parameters: 11178564
Model Weights: 44,714,256 bytes
Model Buffers: 38,560 bytes
Total Size: 43703.92 KB


44752816

In [8]:
teacher_model = train_model(
    teacher_model,
    train_loader,
    val_loader,
    epochs=10,
    lr=LR,
    save_name="../models/teacher",
    device=DEVICE
)

Training: 100%|██████████| 41/41 [06:26<00:00,  9.43s/it]


Epoch 1 | Train Loss 0.0904 F1 0.97 | Val Loss 0.1480 F1 0.9556


Training: 100%|██████████| 41/41 [06:27<00:00,  9.46s/it]


Epoch 2 | Train Loss 0.1024 F1 0.98 | Val Loss 0.1154 F1 0.9518


Training: 100%|██████████| 41/41 [06:32<00:00,  9.58s/it]


...

Epoch 8 | Train Loss 0.0814 F1 0.98 | Val Loss 0.0319 F1 0.9930


Training: 100%|██████████| 41/41 [05:31<00:00,  8.10s/it]


Epoch 9 | Train Loss 0.0594 F1 0.98 | Val Loss 0.0046 F1 1.0000


Training: 100%|██████████| 41/41 [05:55<00:00,  8.67s/it]


Epoch 10 | Train Loss 0.0693 F1 0.98 | Val Loss 0.0078 F1 0.9965


In [9]:
teacher_loss, teacher_f1 = validate(teacher_model, test_loader, device=DEVICE)

print("Teacher F1:", teacher_f1)

Teacher F1: 0.9906324599152211


## Knowledge Distillation

We train the MicroPlant model using guidance from the teacher model.

The student learns from:
- Ground truth labels
- Teacher predictions (soft targets)

In [10]:
student_model = get_microplant(num_classes=len(class_names)).to(DEVICE)

In [12]:
student_model = train_model(
    student_model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    teacher=teacher_model,
    kd_alpha=0.7,
    kd_temp=2.0,
    lr=LR,
    save_name="../models/microplant_kd",
    device=DEVICE
)

Training: 100%|██████████| 41/41 [02:27<00:00,  3.60s/it]


Epoch 1 | Train Loss 3.3934 F1 0.52 | Val Loss 1.0855 F1 0.4367


Training: 100%|██████████| 41/41 [02:31<00:00,  3.69s/it]


Epoch 2 | Train Loss 2.4299 F1 0.64 | Val Loss 0.5607 F1 0.8068


Training: 100%|██████████| 41/41 [02:11<00:00,  3.21s/it]


...

Epoch 8 | Train Loss 0.6395 F1 0.92 | Val Loss 0.1289 F1 0.9513


Training: 100%|██████████| 41/41 [02:13<00:00,  3.27s/it]


Epoch 9 | Train Loss 0.5463 F1 0.93 | Val Loss 0.1296 F1 0.9429


Training: 100%|██████████| 41/41 [02:06<00:00,  3.09s/it]


Epoch 10 | Train Loss 0.5346 F1 0.94 | Val Loss 0.0915 F1 0.9537


In [13]:
student_loss, student_f1 = validate(student_model, test_loader, device=DEVICE)

print("Student (KD) F1:", student_f1)

Student (KD) F1: 0.9528606063859291


## Model Comparison

We compare the performance of:
- Baseline MicroPlant
- Teacher (ResNet18)
- Distilled MicroPlant

In [17]:
print("=== Final Results ===")
print(f"Baseline F1: {baseline_f1:.4f}")
print(f"Teacher F1:  {teacher_f1:.4f}")
print(f"Student F1:  {student_f1:.4f}")

=== Final Results ===
Baseline F1: 0.9344
Teacher F1:  0.9906
Student F1:  0.9529


## Observations

- The teacher model achieves the highest performance due to its capacity
- The distilled student improves over the baseline model
- Knowledge distillation helps transfer knowledge into a smaller model

### Conclusion
The MicroPlant model benefits from distillation, achieving strong performance while remaining lightweight and suitable for edge deployment.